## IMPORT

In [ ]:
import pandas as pd

raw_data = pd.read_csv('../data/employee_attrition_2026_master.csv')
dictionary = pd.read_csv('../data/data_dictionary.csv')

## Checking Missing / Duplicate / raw_data Type

In [ ]:
raw_data.head()

In [ ]:
raw_data.info()

print(f"duplicated = {raw_data.duplicated().sum()}")
print(f"size = {raw_data.size}")
print(f"shape = {raw_data.shape}\n")

## cleaning

|  | Column | Reason |
| :--- | :--- | :--- |
| 1 | employee_id | Unused identifier |
| 2 | attrition_reason | Not relevant for calculation |
| 3 | is_manager | Redundant with *job_level* |
| 4 | generation | Multicollinearity with *age* |


In [ ]:
raw_data = raw_data.drop(columns=["employee_id", "attrition_reason", "is_manager","generation"])

### Delete data for individuals who started working before the age of 16.

In [ ]:
bad_mask = raw_data["tenure_years"] > (raw_data["age"] - 16)
n_bad = bad_mask.sum()

clean_data = raw_data[~bad_mask].reset_index(drop=True)


In [ ]:
clean_data.info()

## Outlier 

In [ ]:
# 1. Check statistical outliers using IQR (continuous columns only)
cols = ["age", "tenure_years", "salary_usd", "commute_minutes", "weekly_hours"]

for col in cols:
    q1, q3 = clean_data[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = ~clean_data[col].between(lo, hi)
    n_outliers = outliers.sum()
    print(f"{col:16s} bounds=({lo:.1f}, {hi:.1f}) outliers={n_outliers} max={clean_data[col].max()}")

# 2. Check logical conflict: remote workers required in office
bad_remote = (clean_data["work_arrangement"] == "remote") & (clean_data["days_in_office_required"] > 0)
print(f"\nRemote conflict rows: {bad_remote.sum()}")

# 3. Check salary logic by job level
print("\nSalary range by job level:")
print(clean_data.groupby("job_level")["salary_usd"].agg(["min", "max"]))

In [ ]:
# 1. Fix logical conflict: remote workers require 0 office days
clean_data.loc[clean_data["work_arrangement"] == "remote", "days_in_office_required"] = 0

# 2. Cap extreme outliers (1st to 99th percentile) without dropping rows
clip_cols = ["salary_usd", "tenure_years", "commute_minutes"]
for col in clip_cols:
    lo, hi = clean_data[col].quantile([0.01, 0.99])
    clean_data[col] = clean_data[col].clip(lo, hi)

print(f"Done, shape: {clean_data.shape}")

In [ ]:
# 1. Verify remote conflicts resolved
conflict_mask = (clean_data["work_arrangement"] == "remote") & (clean_data["days_in_office_required"] > 0)
print(f"Remaining remote conflicts: {conflict_mask.sum()}")

# 2. Check min/max values after clipping
check_cols = ["salary_usd", "tenure_years", "commute_minutes"]
print(clean_data[check_cols].describe().loc[["min", "max"]])

# encoding 

|  | Column | Encoding Method | Reason | 
| :--- | :--- | :--- | :--- |
| 1 | job_level | OrdinalEncoder | Unused identifier |
| 2 | ai_order | OrdinalEncoder |Nominal categorical variable with no inherent order |
| 3 | gender | OneHotEncoder |Nominal categorical variable with no inherent order |
| 4 | department | OneHotEncoder | Nominal categorical variable with no inherent order |
| 5 | work_arrangement | OneHotEncoder | Nominal categorical variable with no inherent order |


In [ ]:
# OrdinalEncoder
job_level_order = ["Junior", "Mid", "Senior", "Lead", "Manager", "Director"]
clean_data["job_level"] = pd.Categorical(clean_data["job_level"], categories=job_level_order, ordered=True).codes
clean_data["job_level"].value_counts().sort_index()

In [ ]:
# OrdinalEncoder
ai_order = ["none", "light", "regular", "power"]
clean_data["ai_tools_adoption"] = pd.Categorical(clean_data["ai_tools_adoption"], categories=ai_order, ordered=True).codes
clean_data["ai_tools_adoption"].value_counts().sort_index()

In [ ]:
# OneHotEncoder
nominal_cols = ["gender", "department", "work_arrangement"]
clean_data = pd.get_dummies(clean_data, columns=nominal_cols ,drop_first=True)

In [ ]:
clean_data.info()

## Summary
---  
### Data Preprocessing Summary
* **Initial Data** : 45,000 rows, 29 columns  
* **Cleaned Data** : 43,439 rows, 34 columns  

#### 1. Data Cleaning
* **Dropped Columns** : employee_id , attrition_reason, is_manager, generation
* **Logic Error Removed** : Filtered out records where tenure_years > (age - 16) (1,561 rows)
* **Logic Conflict Fixed** : Set days_in_office_required = 0 for remote workers
* **Outlier Handling** : Applied percentile clipping to salary_usd, tenure_years, and commute_minutes

#### 2. Feature Encoding  
* **Ordinal Encoding** :
* job_level: Junior < Mid < Senior < Lead < Manager < Director
* ai_tools_adoption: none < light < regular < power
* **One-Hot Encoding (drop_first=True)**:
* gender, department, work_arrangement